# Batch velocity-PSD post-processing of PIVlab runs

Companion to `batch_KineticEnergy.ipynb`. Loops over every subfolder of
`BASE_DIR`, loads `PIVlab_results_uncalibrated.mat`, calibrates the field, crops
to the fixed ROI, and for **each grid point** in the ROI estimates the power
spectral density (Welch) of the two velocity components `U(t)` and `V(t)`.
The per-point PSDs are then **averaged over the whole ROI**.

- Per run it writes `<run>/PostProcessing/VelocityPSD.npz`
  (`f`, `psd_u`, `psd_v`, `psd_total = psd_u + psd_v`, all averaged over the ROI).
- Across all runs it writes `VelocityPSD_summary.csv` / `.xlsx` at `BASE_DIR`
  (tracker of processed runs, like `KineticEnergy_summary`).
- Final figure `VelocityPSD_colormap_vs_fstar.png`: a colormap of the ROI-averaged
  PSD over all runs — **x = f\* = f_lib/f_rot**, **y = PSD frequency**, colour = PSD.

**Duplicate runs**: when several runs share the same parameters (same
`frot/flib/dphi`), only the *second* acquisition is kept in the colormap
(i.e. the `SSn` with the highest index — `SS2` beats `SS1`).

Calibration is read from the **last row** of `acquisition_log.txt`
(`dt_vel = pulse_sep`, `fps = cam_fps`); `xscale = yscale = 1.2323e-4 m/px`.

Set the config in the next cell, then *Run All*. To test on a couple of folders
first, list their names in `ONLY_RUNS`.

In [55]:
import os
import re
import glob
import numpy as np
import pandas as pd
from scipy.io import loadmat
from scipy.signal import welch
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

%matplotlib qt

In [56]:
# --- Configuration --------------------------------------------------------- #
BASE_DIR = ("/Users/jeromenoir/Documents/MyDocuments/LOCAL_PROJECT/"
            "TOPOGRAPHY_LIBRATION/CylinderExperimentsGMA/k6_TopBottom")

PIV_FILENAME = "PIVlab_results_uncalibrated.mat"
LOG_FILENAME = "acquisition_log.txt"
RESULT_FILENAME = "VelocityPSD.npz"          # per-run output (in PostProcessing/)

# Only runs without an existing result are processed unless this is True.
REPROCESS_ALL = True

# Restrict processing to these run-folder names (for testing). Empty -> all runs.
ONLY_RUNS = []

# Pixel calibration (notebook value). Same in x and y.
XSCALE = 1.2323e-4   # m/px
YSCALE = XSCALE

# Fixed ROI, calibrated (metres): [(x0, y0), (x1, y1)] (opposite corners).
PTS_ROI = [(0.02002585173778408, 0.13110555228124998),
           (0.19561960104659090, 0.00659505254715910)]

# PIVlab pairs images (1+2, 3+4, ...), so every velocity field consumes this
# many camera frames. The PIV field sampling frequency is therefore
# f_piv = cam_fps / FRAMES_PER_FIELD (cam_fps in the log is the camera frame
# rate, i.e. TWICE the rate at which PIV fields are produced).
FRAMES_PER_FIELD = 2

# If the acquisition log cannot be read, the run stays UNCALIBRATED: velocity
# dt, PIV sampling and both spatial scales fall back to 1, so velocities are in
# px/frame, positions in px, and timestamps in frame index.
UNCAL_DT = 1.0
UNCAL_FPS = 1.0
UNCAL_SCALE = 1.0

# Folder-name -> physical value conversions (Hz). Folder tokens are integers,
# e.g. 'frot050' -> 0.50 Hz, 'flib0400' -> 0.400 Hz, 'flib1500' -> 1.500 Hz.
FROT_DIVISOR = 100.0
FLIB_DIVISOR = 1000.0

# --- PSD (Welch) settings -------------------------------------------------- #
# Per-point PSD of U(t) and V(t). Welch averages overlapping windowed segments
# (length PSD_NPERSEG, 50% overlap) -> smoother, lower-variance one-sided PSD.
PSD_WINDOW = "hann"
PSD_NPERSEG = None      # None -> full series length (max resolution, higher variance);
                        # an int (e.g. 256) gives a smoother estimate.
PSD_DETREND = "constant"   # remove per-segment mean (kills the DC term)

# --- Colormap settings ----------------------------------------------------- #
# y-axis frequency unit: physical Hz (False) or normalised f/f_rot (True).
NORMALIZE_FREQ_BY_FROT = True
PLOT_NFREQ = 400        # rows of the common frequency grid for the colormap
PLOT_FMAX = None        # y-axis max (Hz, or f/f_rot if normalised). None -> auto
PLOT_CMAP = "viridis"

In [57]:
def load_piv(file_path):
    """Load a PIVlab wienerwurst .mat file and return calibrated-ready fields.

    Returns X, Y (2D grids), U, V (validated velocity, NaN where invalid),
    and nframes. Axis flips / sign flips reproduce the notebook exactly.
    """
    mat = loadmat(file_path,
                  variable_names=["x", "y", "u", "v", "u_filt", "v_filt"])
    if "u_filt" not in mat:
        raise ValueError("Not a wienerwurst PIV file: %s" % file_path)

    X_original = mat["x"][:, :, 0]
    Y_original = mat["y"][:, :, 0]

    U_filtered = mat["u_filt"]
    V_filtered = mat["v_filt"]
    U_org = mat["u"]           # velocity prior to validation
    V_org = mat["v"]

    # Flip axes (match notebook)
    X = X_original[:, ::-1].astype(float, copy=False)
    Y = Y_original[::-1, :].astype(float, copy=False)
    U = U_filtered[::-1, ::-1, :].astype(float, copy=False)
    V = V_filtered[::-1, ::-1, :].astype(float, copy=False)
    U_original = U_org[::-1, ::-1, :]
    V_original = V_org[::-1, ::-1, :]

    nframes = U_filtered.shape[2]

    # Mask wherever the un-validated vectors are NaN
    mask = np.isnan(U_original) | np.isnan(V_original)
    U[mask] = np.nan
    V[mask] = np.nan

    Y = Y.max() - Y
    V = -V

    return X, Y, U, V, nframes


def create_mask(x, y, pts_roi):
    """Boolean mask of grid points inside the ROI rectangle."""
    pts = np.asarray(pts_roi).reshape(2, 2)
    (x1, y1), (x2, y2) = pts
    xmin, xmax = sorted((x1, x2))
    ymin, ymax = sorted((y1, y2))
    return (x >= xmin) & (x <= xmax) & (y >= ymin) & (y <= ymax)


def extract_roi(x, y, u, v, mask):
    """Crop x, y (2D) and u, v (3D, frames last) to the ROI bounding box."""
    if not np.any(mask):
        return (np.array([]),) * 4
    rows = np.any(mask, axis=1)
    cols = np.any(mask, axis=0)
    x_roi = x[np.ix_(rows, cols)]
    y_roi = y[np.ix_(rows, cols)]
    u_roi = u[np.ix_(rows, cols, np.arange(u.shape[2]))]
    v_roi = v[np.ix_(rows, cols, np.arange(v.shape[2]))]
    return x_roi, y_roi, u_roi, v_roi

In [58]:
def read_acquisition_params(log_path):
    """Return (dt_vel_s, fps_Hz, ok) from the run's acquisition log.

    The log is tab-separated with a header; the recording row is the LAST data
    row. From it:
      - dt_vel = pulse_sep * 1e-6            (s, image separation within a pair
                                              -> velocity magnitude)
      - fps    = cam_fps / FRAMES_PER_FIELD  (Hz, PIV field sampling frequency ->
                                              timestamps). cam_fps is the camera
                                              frame rate; PIV pairs images, so a
                                              field is produced every 2 frames.
    Returns ok=False if the log / its last row cannot be read.
    """
    try:
        with open(log_path) as fh:
            lines = [ln.rstrip("\n") for ln in fh if ln.strip()]
        header = lines[0].split("\t")
        i_pulse = header.index("pulse_sep")
        i_fps = header.index("cam_fps")
        for ln in reversed(lines[1:]):
            cols = ln.split("\t")
            try:
                return (float(cols[i_pulse]) * 1e-6,
                        float(cols[i_fps]) / FRAMES_PER_FIELD, True)
            except (IndexError, ValueError):
                continue
    except (OSError, ValueError):
        pass
    return UNCAL_DT, UNCAL_FPS, False


def parse_run_name(name):
    """Parse frot/flib (Hz) and dphi (deg) from a folder name.

    e.g. 'frot050_flib0400_dphi2.5deg_SS1' -> (0.5, 0.4, 2.5).
    Missing tokens come back as NaN.
    """
    frot = re.search(r"frot(\d+)", name)
    flib = re.search(r"flib(\d+)", name)
    dphi = re.search(r"dphi([\d.]+)deg", name)
    frot_hz = float(frot.group(1)) / FROT_DIVISOR if frot else np.nan
    flib_hz = float(flib.group(1)) / FLIB_DIVISOR if flib else np.nan
    dphi_deg = float(dphi.group(1)) if dphi else np.nan
    return frot_hz, flib_hz, dphi_deg

In [59]:
def _interp_nan_rows(A):
    """Linearly interpolate NaNs along time for each row (point) of A.

    A has shape (npoints, nframes). Rows that are entirely NaN are flagged in the
    returned `keep` mask (they are excluded from the ROI average). Interior /
    edge NaNs are filled by linear interpolation (edges use the nearest value).
    """
    A = A.astype(float, copy=True)
    n = A.shape[1]
    idx = np.arange(n)
    keep = np.ones(A.shape[0], dtype=bool)
    for i in range(A.shape[0]):
        row = A[i]
        m = np.isnan(row)
        if m.all():
            keep[i] = False
            continue
        if m.any():
            row[m] = np.interp(idx[m], idx[~m], row[~m])
    return A, keep


def averaged_psd(u_roi, v_roi, fps, window=PSD_WINDOW, nperseg=PSD_NPERSEG,
                 detrend=PSD_DETREND):
    """ROI-averaged Welch PSD of U(t) and V(t).

    For every grid point in the ROI the one-sided PSD of its time series is
    estimated with scipy.signal.welch, then averaged over all valid points.
    Returns (f, psd_u, psd_v, psd_total, npoints) where psd_total = psd_u+psd_v.
    Units: (m/s)^2 / Hz. Returns NaNs if no valid point exists.
    """
    nf = u_roi.shape[-1]
    U = u_roi.reshape(-1, nf)
    V = v_roi.reshape(-1, nf)
    U, keepU = _interp_nan_rows(U)
    V, keepV = _interp_nan_rows(V)
    nseg = int(nperseg) if nperseg else nf
    nseg = max(1, min(nseg, nf))

    if not keepU.any() or not keepV.any():
        f = welch(np.zeros(nf), fs=fps, window=window, nperseg=nseg,
                  detrend=detrend)[0]
        nanv = np.full_like(f, np.nan, dtype=float)
        return f, nanv, nanv.copy(), nanv.copy(), 0

    f, Pu = welch(U[keepU], fs=fps, window=window, nperseg=nseg,
                  detrend=detrend, scaling="density", axis=-1)
    _, Pv = welch(V[keepV], fs=fps, window=window, nperseg=nseg,
                  detrend=detrend, scaling="density", axis=-1)
    psd_u = np.nanmean(Pu, axis=0)
    psd_v = np.nanmean(Pv, axis=0)
    psd_total = psd_u + psd_v
    npoints = int(keepU.sum())
    return f, psd_u, psd_v, psd_total, npoints

In [60]:
def build_row(name, out_file, processed, fps=np.nan, nframes=np.nan,
              npoints=np.nan, f_peak=np.nan, dt_vel=np.nan, ok=np.nan):
    """Assemble one summary-table row (folder-name metadata + results/flags)."""
    frot_hz, flib_hz, dphi_deg = parse_run_name(name)
    fstar = flib_hz / frot_hz if frot_hz else np.nan
    # Session index: the integer following 'SS' in the folder name (SS2 -> 2).
    ss = re.search(r"SS(\d+)", name)
    run_idx = int(ss.group(1)) if ss else np.nan
    return {"run": name, "run idx": run_idx, "processed": processed,
            "frot_Hz": frot_hz, "flib_Hz": flib_hz, "dphi_deg": dphi_deg,
            "fstar": fstar, "calibrated": ok, "dt_vel_s": dt_vel, "fps_Hz": fps,
            "nframes": nframes, "npoints": npoints, "f_peak_Hz": f_peak,
            "npz": out_file}


def _peak_freq(f, psd):
    """Frequency of the PSD maximum, ignoring the DC bin. NaN if unavailable."""
    if f.size < 2 or np.all(np.isnan(psd)):
        return np.nan
    j = np.nanargmax(psd[1:]) + 1
    return float(f[j])


def process_run(run_dir, reprocess=False):
    """Process one run folder. Returns a summary dict.

    If a per-run result .npz already exists and reprocess is False, the row is
    rebuilt from that cache (the large .mat is not re-read). A folder with no
    PIV file yields a row with processed=False and NaN metrics.
    """
    name = os.path.basename(run_dir.rstrip("/"))
    piv_file = os.path.join(run_dir, PIV_FILENAME)
    out_dir = os.path.join(run_dir, "PostProcessing")
    out_file = os.path.join(out_dir, RESULT_FILENAME)

    # Reuse an existing result unless a reprocess was requested.
    if os.path.isfile(out_file) and not reprocess:
        try:
            d = np.load(out_file, allow_pickle=True)
            print("  [skip] %-40s already processed (cached)" % name)
            return build_row(name, out_file, True,
                             fps=float(d["fps"]), nframes=int(d["nframes"]),
                             npoints=int(d["npoints"]),
                             f_peak=float(d["f_peak"]),
                             dt_vel=float(d["dt_vel"]), ok=bool(d["calibrated"]))
        except Exception as exc:
            print("  [warn] %s: cached result unreadable (%s) -> reprocessing"
                  % (name, exc))

    if not os.path.isfile(piv_file):
        print("  [skip] no %s in %s" % (PIV_FILENAME, name))
        return build_row(name, "", False)

    dt_vel, fps, ok = read_acquisition_params(os.path.join(run_dir, LOG_FILENAME))
    if ok:
        xscale, yscale = XSCALE, YSCALE
    else:
        print("  [warn] %s: Calibration not possible - all velocities will be "
              "in px/frame" % name)
        xscale = yscale = UNCAL_SCALE   # dt_vel, fps already fell back to 1

    X, Y, U, V, nframes = load_piv(piv_file)

    # Calibrate: positions -> m (or px), velocities -> m/s (or px/frame).
    X = xscale * X
    Y = yscale * Y
    U = xscale * U / dt_vel
    V = yscale * V / dt_vel

    # Crop to the fixed ROI.
    mask = create_mask(X, Y, PTS_ROI)
    Xr, Yr, Ur, Vr = extract_roi(X, Y, U, V, mask)
    if Ur.size == 0:
        print("  [warn] ROI empty for %s -> using full field" % name)
        Xr, Yr, Ur, Vr = X, Y, U, V

    # ROI-averaged PSD of the two velocity components. PIV fields are produced
    # at f_piv = cam_fps / FRAMES_PER_FIELD (image pairing), so `fps` here is
    # already that field rate and is the PSD sampling frequency.
    f, psd_u, psd_v, psd_total, npoints = averaged_psd(Ur, Vr, fps)
    f_peak = _peak_freq(f, psd_total)

    # Save per-run npz.
    os.makedirs(out_dir, exist_ok=True)
    np.savez(out_file,
             run=name, PIV_file=piv_file, calibrated=ok,
             dt_vel=dt_vel, fps=fps, xscale=xscale, yscale=yscale,
             pts_ROI=np.array(PTS_ROI), nframes=nframes, npoints=npoints,
             window=PSD_WINDOW, nperseg=(PSD_NPERSEG or nframes),
             f=f, psd_u=psd_u, psd_v=psd_v, psd_total=psd_total, f_peak=f_peak)

    print("  [ok] %-40s nframes=%d fps=%.4gHz npts=%d  f_peak=%.4gHz"
          % (name, nframes, fps, npoints, f_peak))

    return build_row(name, out_file, True, fps=fps, nframes=nframes,
                     npoints=npoints, f_peak=f_peak, dt_vel=dt_vel, ok=ok)

In [61]:
def _edges(centers):
    """Cell edges around 1-D (possibly non-uniform) centers for pcolormesh.

    A single center gets a small symmetric width so a one-column colormap still
    renders (the degenerate case that `shading='nearest'` cannot draw).
    """
    c = np.asarray(centers, dtype=float)
    if c.size == 1:
        w = abs(c[0]) * 0.05 or 1.0
        return np.array([c[0] - w, c[0] + w])
    mid = 0.5 * (c[:-1] + c[1:])
    return np.concatenate([[c[0] - (mid[0] - c[0])], mid,
                           [c[-1] + (c[-1] - mid[-1])]])


def mark_kept(df):
    """Flag which processed run to keep per (frot, flib, dphi) group.

    Duplicate acquisitions of the same parameters are collapsed to the one with
    the highest 'SSn' index (SS2 beats SS1). Runs with no SS token count as 1.
    Adds a boolean 'kept' column (True for the retained run in each group).
    """
    df = df.copy()
    df["kept"] = False
    proc = df[df["processed"] == True].copy()
    if proc.empty:
        return df
    proc["_ss"] = proc["run idx"].fillna(1)
    for _, g in proc.groupby(["frot_Hz", "flib_Hz", "dphi_deg"], dropna=False):
        best = g["_ss"].idxmax()
        df.loc[best, "kept"] = True
    return df


def plot_colormap(df, base_dir):
    """Colormap of the ROI-averaged PSD across all kept runs.

    x = f* = f_lib/f_rot (one column per run), y = PSD frequency
    (Hz, or f/f_rot if NORMALIZE_FREQ_BY_FROT), colour = PSD (log scale).
    Each run's PSD is interpolated onto a shared frequency grid.
    """
    d = df[(df["kept"] == True) & (df["processed"] == True)]
    d = d.dropna(subset=["fstar"]).sort_values("fstar")
    if d.empty:
        print("  (colormap skipped: no kept runs with valid f*)")
        return None

    runs = []   # (fstar, freq_axis, psd_total)
    for _, r in d.iterrows():
        try:
            z = np.load(r["npz"], allow_pickle=True)
        except Exception as exc:
            print("  [warn] cannot load %s (%s)" % (r["npz"], exc))
            continue
        f = np.asarray(z["f"], dtype=float)
        p = np.asarray(z["psd_total"], dtype=float)
        if f.size < 2 or np.all(np.isnan(p)):
            continue
        yf = f / r["frot_Hz"] if (NORMALIZE_FREQ_BY_FROT and r["frot_Hz"]) else f
        runs.append((float(r["fstar"]), yf, p))

    if not runs:
        print("  (colormap skipped: no usable PSD arrays)")
        return None

    fstars = np.array([a for a, _, _ in runs])
    fmax = PLOT_FMAX if PLOT_FMAX else min(yf.max() for _, yf, _ in runs)
    grid = np.linspace(0.0, fmax, PLOT_NFREQ)

    Z = np.full((PLOT_NFREQ, len(runs)), np.nan)
    for j, (_, yf, p) in enumerate(runs):
        Z[:, j] = np.interp(grid, yf, p, left=np.nan, right=np.nan)

    Zpos = np.where(Z > 0, Z, np.nan)
    finite = Zpos[np.isfinite(Zpos)]
    if finite.size == 0:
        print("  (colormap skipped: PSD all non-positive)")
        return None
    vmin = np.nanpercentile(finite, 5)
    vmax = np.nanpercentile(finite, 99.5)
    if not (vmin > 0):
        vmin = finite.min()

    x_edges = _edges(fstars)
    y_edges = _edges(grid)
    fig, ax = plt.subplots(figsize=(11, 6))
    pcm = ax.pcolormesh(x_edges, y_edges, Zpos, shading="flat",
                        cmap=PLOT_CMAP, norm=LogNorm(vmin=vmin, vmax=vmax))
    cbar = fig.colorbar(pcm, ax=ax)
    cbar.set_label(r"ROI-averaged PSD  ($\mathrm{m^2\,s^{-2}\,Hz^{-1}}$)",
                   fontsize=12)
    ax.set_xlabel(r"$f^* = f_{\mathrm{lib}} / f_{\mathrm{rot}}$", fontsize=13)
    ylabel = (r"$f / f_{\mathrm{rot}}$" if NORMALIZE_FREQ_BY_FROT
              else "frequency (Hz)")
    ax.set_ylabel(ylabel, fontsize=13)
    ax.set_title("ROI-averaged velocity PSD vs. $f^*$", fontsize=14)

    # Mark the run f* positions along the top.
    ax.scatter(fstars, np.full_like(fstars, grid[-1]), marker="v", s=18,
               color="w", edgecolor="k", linewidth=0.4, clip_on=False, zorder=5)

    fig.tight_layout()
    out = os.path.join(base_dir, "VelocityPSD_colormap_vs_fstar.png")
    fig.savefig(out, dpi=200, bbox_inches="tight")
    print("Colormap written to:\n  %s" % out)
    return fig

In [62]:
def run_batch(reprocess_all=REPROCESS_ALL, only_runs=None):
    """Process runs under BASE_DIR, write the summary + colormap, return df.

    only_runs (list of folder names) restricts processing for testing; if None
    it falls back to the ONLY_RUNS config (empty -> all runs).
    """
    if only_runs is None:
        only_runs = ONLY_RUNS
    subdirs = sorted(dd for dd in glob.glob(os.path.join(BASE_DIR, "*"))
                     if os.path.isdir(dd)
                     and not os.path.basename(dd).startswith("."))
    if only_runs:
        wanted = set(only_runs)
        subdirs = [dd for dd in subdirs if os.path.basename(dd) in wanted]

    print("Found %d subfolders to process in %s%s"
          % (len(subdirs), BASE_DIR,
             "  (reprocessing ALL)" if reprocess_all else
             "  (skipping already-processed)"))

    rows = []
    for run_dir in subdirs:
        try:
            row = process_run(run_dir, reprocess=reprocess_all)
        except Exception as exc:  # keep the batch going
            print("  [error] %s: %s" % (os.path.basename(run_dir), exc))
            row = None
        if row is not None:
            rows.append(row)

    if not rows:
        print("No runs found.")
        return None

    df = pd.DataFrame(rows).sort_values(["frot_Hz", "flib_Hz", "dphi_deg"])
    df = mark_kept(df)

    csv_path = os.path.join(BASE_DIR, "VelocityPSD_summary.csv")
    df.to_csv(csv_path, index=False)
    n_proc = int((df["processed"] == True).sum())
    n_kept = int((df["kept"] == True).sum())
    print("\nSummary (%d runs: %d processed, %d kept for colormap) written to:\n  %s"
          % (len(df), n_proc, n_kept, csv_path))
    try:
        xlsx_path = os.path.join(BASE_DIR, "VelocityPSD_summary.xlsx")
        df.to_excel(xlsx_path, index=False)
        print("  %s" % xlsx_path)
    except Exception as exc:
        print("  (xlsx skipped: %s)" % exc)

    plot_colormap(df, BASE_DIR)
    return df

## Run

`ONLY_RUNS` in the config cell restricts the batch to a few folders for testing.
Clear it (`ONLY_RUNS = []`) and re-run to process **all** runs. Set
`REPROCESS_ALL = True` (or call `run_batch(reprocess_all=True)`) to recompute
every run from the `.mat` files.

In [63]:
df = run_batch(reprocess_all=REPROCESS_ALL)
df.head(40)


Found 33 subfolders to process in /Users/jeromenoir/Documents/MyDocuments/LOCAL_PROJECT/TOPOGRAPHY_LIBRATION/CylinderExperimentsGMA/k6_TopBottom  (reprocessing ALL)
  [ok] frot050_flib0400_dphi2deg_SS1            nframes=1500 fps=25Hz npts=5607  f_peak=0.4Hz
  [ok] frot050_flib0400_dphi2deg_SS2            nframes=1500 fps=25Hz npts=5607  f_peak=0.4Hz
  [ok] frot050_flib0405_dphi2deg_SS1            nframes=1500 fps=25Hz npts=5607  f_peak=0.4Hz
  [ok] frot050_flib0410_dphi2deg_SS1            nframes=1500 fps=25Hz npts=5607  f_peak=0.4167Hz
  [ok] frot050_flib0415_dphi2deg_SS1            nframes=1500 fps=25Hz npts=5607  f_peak=0.4167Hz
  [ok] frot050_flib0420_dphi2deg_SS1            nframes=1500 fps=25Hz npts=5607  f_peak=0.4167Hz
  [ok] frot050_flib0425_dphi2deg_SS1            nframes=1500 fps=25Hz npts=5607  f_peak=0.4333Hz
  [ok] frot050_flib0430_dphi2deg_SS1            nframes=1500 fps=25Hz npts=5607  f_peak=0.4333Hz
  [ok] frot050_flib0435_dphi2deg_SS1            nframes=1500 fps=25H

,run,run idx,processed,frot_Hz,flib_Hz,dphi_deg,fstar,calibrated,dt_vel_s,fps_Hz,nframes,npoints,f_peak_Hz,npz,kept
0,frot050_flib0400_dphi2deg_SS1,1,True,0.5,0.400,2.0,0.80,True,0.020000,25.0,1500,5607,0.400000,/Users/jeromenoir/Documents/MyDocuments/LOCAL_...,False
1,frot050_flib0400_dphi2deg_SS2,2,True,0.5,0.400,2.0,0.80,True,0.020000,25.0,1500,5607,0.400000,/Users/jeromenoir/Documents/MyDocuments/LOCAL_...,True
2,frot050_flib0405_dphi2deg_SS1,1,True,0.5,0.405,2.0,0.81,True,0.020000,25.0,1500,5607,0.400000,/Users/jeromenoir/Documents/MyDocuments/LOCAL_...,True
3,frot050_flib0410_dphi2deg_SS1,1,True,0.5,0.410,2.0,0.82,True,0.020000,25.0,1500,5607,0.416667,/Users/jeromenoir/Documents/MyDocuments/LOCAL_...,True
4,frot050_flib0415_dphi2deg_SS1,1,True,0.5,0.415,2.0,0.83,True,0.020000,25.0,1500,5607,0.416667,/Users/jeromenoir/Documents/MyDocuments/LOCAL_...,True
5,frot050_flib0420_dphi2deg_SS1,1,True,0.5,0.420,2.0,0.84,True,0.020000,25.0,1500,5607,0.416667,/Users/jeromenoir/Documents/MyDocuments/LOCAL_...,True
6,frot050_flib0425_dphi2deg_SS1,1,True,0.5,0.425,2.0,0.85,True,0.020000,25.0,1500,5607,0.433333,/Users/jeromenoir/Documents/MyDocuments/LOCAL_...,True
7,frot050_flib0430_dphi2deg_SS1,1,True,0.5,0.430,2.0,0.86,True,0.020000,25.0,1500,5607,0.433333,/Users/jeromenoir/Documents/MyDocuments/LOCAL_...,True
8,frot050_flib0435_dphi2deg_SS1,1,True,0.5,0.435,2.0,0.87,True,0.020000,25.0,1500,5607,0.433333,/Users/jeromenoir/Documents/MyDocuments/LOCAL_...,True
9,frot050_flib0440_dphi2deg_SS1,1,True,0.5,0.440,2.0,0.88,True,0.020000,25.0,1500,5607,0.433333,/Users/jeromenoir/Documents/MyDocuments/LOCAL_...,True
